<a href="https://colab.research.google.com/github/Maverixk/contrastive-bridge-3d-vlm/blob/main/CV_Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sensor-Aware Contrastive Bridge for Real-World 3D Vision Language Models

## General information

### Students:
- Lorenzo Carlini
- Riccardo Aquilanti
- Luca Di Carlo

### Abstract
(I will copy-paste it at some point, I'm lazy now).

## Imports

### Libraries

Below, you'll find a cell meant for importing all the libraries we need for our project. Before the imports, we want to install some specific packages that we are going to use. More specifically:
- **open-clip-torch**, essential for CLIP text encoder;
- **huggingface_hub**, required to dynamically download stuff from HuggingFace;
- **h5py**, used to handle the dataset we will import.

In [8]:
# ── Packages install ──────────────────────────────────────────────────────────
!pip install -q open-clip-torch huggingface_hub h5py

# ── Standard libraries ────────────────────────────────────────────────────────
import os
import random
import glob
import json
import zipfile
import urllib.request
from collections import OrderedDict

# ── Data management & visualizations ──────────────────────────────────────────
import numpy as np
import h5py
import matplotlib.pyplot as plt

# ── PyTorch ecosystem ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from timm.models.layers import DropPath
except ImportError:
    from timm.layers import DropPath

# ── Deep Learning Utilities ───────────────────────────────────────────────────
import open_clip
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm.notebook import tqdm

# ── Google Colab ──────────────────────────────────────────────────────────────
from google.colab import drive

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


### Setup

In this block, we want to create our directory on **Google Drive**, in order to store the datasets we'll use. Moreover, we intend to setup the environment for the use of **CUDA drivers**, as the training phase will mainly be performed with NVIDIA GPUs.

In [ ]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/SACB'
for d in ['models', 'data/modelnet40', 'data/scanobjectnn', 'checkpoints', 'results']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    AMP_DTYPE = torch.bfloat16 if 'A100' in name or 'A10' in name else torch.float16
    print(f'GPU: {name}  VRAM: {vram:.1f} GB  AMP dtype: {AMP_DTYPE}')
else:
    AMP_DTYPE = torch.float32
    print('WARNING: no GPU')

## Globals

### Constants
The following cell contains all the **constants** that will be used in the rest of this notebook.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ULIP2_CKPT       = f'{BASE}/models/ulip2_pointbert.pt'
MODELNET40_DIR   = f'{BASE}/data/modelnet40'
SCANOBJECTNN_DIR = f'{BASE}/data/scanobjectnn'
CHECKPOINTS_DIR  = f'{BASE}/checkpoints'
RESULTS_DIR      = f'{BASE}/results'

# ── PointBERT config ──────────────────────────────────────────────────────────
NUM_POINTS  = 2048
NUM_GROUP   = 128   # reduced from 512 (8192-pt model) to suit 2048-pt inputs
GROUP_SIZE  = 32
TRANS_DIM   = 384
DEPTH       = 12
DROP_PATH   = 0.1
NUM_HEADS   = 6
ENC_DIMS    = 256
PC_FEAT_DIM = TRANS_DIM * 2   # 768
FEAT_DIM    = 512             # after pc_projection

# ── Training ──────────────────────────────────────────────────────────────────
SEED          = 42
NUM_EPOCHS    = 30
BATCH_SIZE    = 32   # reduce to 16 on free T4 if OOM
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
TEMPERATURE   = 0.07

# ── Noise params ──────────────────────────────────────────────────────────────
KINECT_AXIAL_SIGMA   = 0.0012
KINECT_LATERAL_SIGMA = 0.0019
KINECT_DROPOUT_P     = 0.05
GAUSSIAN_SIGMA       = 0.02

# ── Class labels ──────────────────────────────────────────────────────────────
SCANOBJECTNN_CLASSES = [
    'bag','bin','box','cabinet','chair',
    'desk','display','door','shelf','table',
    'bed','pillow','sink','sofa','toilet',
]

MN40_CLASSES = [
    'airplane','bathtub','bed','bench','bookshelf','bottle','bowl','car','chair',
    'cone','cup','curtain','desk','door','dresser','flower pot','glass box',
    'guitar','keyboard','lamp','laptop','mantel','monitor','night stand','person',
    'piano','plant','radio','range hood','sink','sofa','stairs','stool','table',
    'tent','toilet','tv stand','vase','wardrobe','xbox',
]

# Richer descriptions to avoid text-space degeneracy (e.g. desk/table similarity)
CLASS_DESCS = {
    'bag':'bag','bin':'trash bin','box':'cardboard box',
    'cabinet':'storage cabinet','chair':'chair','desk':'office desk',
    'display':'computer monitor','door':'door','shelf':'bookshelf',
    'table':'dining table','bed':'bed','pillow':'pillow',
    'sink':'sink','sofa':'sofa couch','toilet':'toilet',
}

PROMPT_TEMPLATES = [
    'a 3D scan of a {}',
    'a point cloud of a {}',
    'a {} in an indoor scene',
    '{}',
]

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('Constants loaded.')

## Utils

### Point cloud utilities
Below we will include the **utility functions** meant for dealing with the **point clouds**.

In [ ]:
# ── FPS (pure PyTorch — no compiled CUDA extension) ───────────────────────────
def _index_points(points, idx):
    B = points.shape[0]
    view = [1] * idx.dim(); view[0] = B
    bidx = torch.arange(B, device=points.device).view(view).expand_as(idx)
    return points[bidx, idx, :]

def fps(xyz, npoint):
    """Furthest Point Sampling — (B,N,3) -> (B,npoint,3)"""
    B, N, _ = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=xyz.device)
    dist      = torch.full((B, N), 1e10, device=xyz.device)
    farthest  = torch.randint(0, N, (B,), device=xyz.device)
    bidx      = torch.arange(B, device=xyz.device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[bidx, farthest].unsqueeze(1)   # B 1 3
        d = ((xyz - centroid) ** 2).sum(-1)
        dist = torch.minimum(dist, d)
        farthest = dist.argmax(-1)
    return _index_points(xyz, centroids)

def _square_dist(src, dst):
    B, N, _ = src.shape; _, M, _ = dst.shape
    d  = -2 * torch.matmul(src, dst.permute(0,2,1))
    d +=  src.pow(2).sum(-1).view(B, N, 1)
    d +=  dst.pow(2).sum(-1).view(B, 1, M)
    return d

def knn_point(nsample, xyz, new_xyz):
    """(B,N,3),(B,S,3) -> (B,S,nsample) indices"""
    _, idx = torch.topk(_square_dist(new_xyz, xyz), nsample, dim=-1, largest=False, sorted=False)
    return idx

# ── Grouping + mini-encoder (PointBERT tokeniser) ─────────────────────────────
class Group(nn.Module):
    def __init__(self, num_group, group_size):
        super().__init__()
        self.num_group  = num_group
        self.group_size = group_size

    def forward(self, xyz):
        B, N, _ = xyz.shape
        center   = fps(xyz, self.num_group)                      # B G 3
        idx      = knn_point(self.group_size, xyz, center)       # B G M
        idx_flat = (idx + torch.arange(B, device=xyz.device).view(-1,1,1)*N).view(-1)
        nb = xyz.view(B*N, 3)[idx_flat].view(B, self.num_group, self.group_size, 3)
        nb = nb - center.unsqueeze(2)                            # local coords
        return nb, center

class PatchEncoder(nn.Module):
    """Per-group PointNet: (B,G,M,3) -> (B,G,encoder_dims)"""
    def __init__(self, encoder_dims):
        super().__init__()
        self.encoder_dims = encoder_dims
        self.first  = nn.Sequential(nn.Conv1d(3,128,1), nn.BatchNorm1d(128), nn.ReLU(True), nn.Conv1d(128,256,1))
        self.second = nn.Sequential(nn.Conv1d(512,512,1), nn.BatchNorm1d(512), nn.ReLU(True), nn.Conv1d(512,encoder_dims,1))

    def forward(self, groups):
        B, G, M, _ = groups.shape
        x  = groups.reshape(B*G, M, 3)
        f  = self.first(x.transpose(2,1))                       # BG 256 M
        fg = f.max(2, keepdim=True)[0]                          # BG 256 1
        f  = self.second(torch.cat([fg.expand(-1,-1,M), f], 1)) # BG D M
        return f.max(2)[0].reshape(B, G, self.encoder_dims)     # B G D

print('Utilities defined.')

### Noise & curvature utilities
Our task requires us to deal with different **types of noise**, so it is optimal to create and group their specific utilities all together.

In [ ]:
def apply_gaussian_noise(pc):
    return pc + torch.randn_like(pc) * GAUSSIAN_SIGMA

def apply_kinect_noise(pc):
    z      = pc[:, :, 2:3].clamp(min=0.1)
    axial  = torch.randn_like(pc[:,:,:1]) * KINECT_AXIAL_SIGMA * z**2
    lat    = torch.randn_like(pc[:,:,:2]) * KINECT_LATERAL_SIGMA * z
    noisy  = pc + torch.cat([lat, axial], dim=-1)
    mask   = torch.rand(pc.shape[0], pc.shape[1], 1, device=pc.device) > KINECT_DROPOUT_P
    return noisy * mask

def batch_curvatures(pc):
    """(B,N,3) -> (B,) curvature proxy via smallest PCA eigenvalue ratio."""
    B, N, _ = pc.shape
    idx = torch.randint(0, N, (B, min(256, N)), device=pc.device)
    sub = pc[torch.arange(B).unsqueeze(1), idx]
    sub = sub - sub.mean(1, keepdim=True)
    cov = sub.transpose(1,2) @ sub / sub.shape[1]
    eigs = torch.linalg.eigvalsh(cov.float()).clamp(min=0)
    return eigs[:,0] / (eigs.sum(-1) + 1e-8)

def sc_hns_mask(curvatures, delta=0.05):
    """(B,) -> (B,B) float mask: 1 where curvature difference < delta (hard negatives)."""
    diff = (curvatures.unsqueeze(1) - curvatures.unsqueeze(0)).abs()
    mask = (diff < delta).float()
    mask.fill_diagonal_(0)
    return mask

print('Noise utilities defined.')

### Training utilities

In [ ]:
def info_nce(bridged, clean, tau):
    b = F.normalize(bridged, dim=1); c = F.normalize(clean, dim=1)
    logits = (b @ c.T) / tau
    return F.cross_entropy(logits, torch.arange(logits.size(0), device=logits.device))

def info_nce_schns(bridged, clean, curvs, tau, delta=0.05):
    b = F.normalize(bridged, dim=1); c = F.normalize(clean, dim=1)
    logits = (b @ c.T) / tau + sc_hns_mask(curvs, delta) / tau
    return F.cross_entropy(logits, torch.arange(logits.size(0), device=logits.device))

class Trainer:
    def __init__(self, variant, encoder):
        assert variant in ('A','B','C')
        self.variant   = variant
        self.encoder   = encoder
        self.use_schns = variant in ('B','C')
        self.bridge    = SensorAwareContrastiveBridge().to(device)
        self.optimizer = optim.AdamW(self.bridge.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
        self.scaler    = torch.amp.GradScaler('cuda')
        self.history   = []
        print(f'[Trainer] variant={variant}  bridge params={self.bridge.count_parameters():,}')

    def train_epoch(self, loader):
        self.bridge.train()
        total = 0.0
        for clean_pc, noisy_pc in tqdm(loader, leave=False):
            clean_pc = clean_pc.to(device, non_blocking=True)
            noisy_pc = noisy_pc.to(device, non_blocking=True)
            with torch.no_grad():
                clean_f = self.encoder.encode_pc(clean_pc)
                noisy_f = self.encoder.encode_pc(noisy_pc)
            curvs = batch_curvatures(clean_pc) if self.use_schns else None
            self.optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                br = self.bridge(noisy_f)
                loss = info_nce_schns(br, clean_f, curvs, TEMPERATURE) if self.use_schns  else info_nce(br, clean_f, TEMPERATURE)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.bridge.parameters(), 1.0)
            self.scaler.step(self.optimizer); self.scaler.update()
            total += loss.item()
        self.scheduler.step()
        avg = total / max(len(loader), 1)
        self.history.append(avg)
        return avg

    def save(self, epoch):
        path = f'{CHECKPOINTS_DIR}/sacb_variant{self.variant}_epoch{epoch:03d}.pt'
        torch.save({'epoch':epoch,'variant':self.variant,
                    'bridge_state':self.bridge.state_dict(),'loss':self.history}, path)
        return path

print('Trainer defined.')

### Evaluation utilities

In [ ]:
@torch.no_grad()
def evaluate(bridge, encoder, loader, classes):
    """Returns (overall_acc %, per_class_acc array, mean_cd)."""
    if loader is None:
        return None, None, None
    descs = [CLASS_DESCS.get(c, c) for c in classes] if classes is SCANOBJECTNN_CLASSES else None
    text_f = encoder.encode_text(classes, descs).to(device)
    preds, lbls, cds = [], [], []
    for pc, lbl in loader:
        pc = pc.to(device)
        clean_f = encoder.encode_pc(pc)
        if bridge is not None:
            br_f = F.normalize(bridge(clean_f), dim=1)
            cds.append((F.normalize(br_f,dim=1) - F.normalize(clean_f,dim=1)).norm(dim=1).mean().item())
            feat = br_f
        else:
            feat = clean_f
        preds.append((feat @ text_f.T).argmax(1).cpu())
        lbls.append(lbl)
    pred  = torch.cat(preds).numpy()
    label = torch.cat(lbls).numpy()
    acc   = (pred == label).mean() * 100
    per_cls = np.array([(pred[label==c]==c).mean()*100 if (label==c).any() else 0.0
                        for c in range(len(classes))])
    cd = float(np.mean(cds)) if cds else 0.0
    return acc, per_cls, cd

def load_bridge(path):
    bridge = SensorAwareContrastiveBridge().to(device).eval()
    ckpt   = torch.load(path, map_location=device, weights_only=False)
    bridge.load_state_dict(ckpt['bridge_state'])
    return bridge

print('Evaluate defined.')

## Data

### Datasets
The underlying cell defines the objects for the 3 datasets we will use:
- **ModelNet40**;
- **ScanObjectNN**;
- **ShapeNetCore**.

Additionally, there are also 2 **support functions** meant for **normalizing** the point clouds and creating the **train and test set** of each dataset.

In [ ]:
def normalize_pc(pc):
    pc = pc - pc.mean(0)
    return pc / (pc.norm(dim=-1).max() + 1e-8)

def _load_obj_vertices(path):
    """Parse XYZ vertices from an OBJ file without extra dependencies."""
    verts = []
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append((float(parts[1]), float(parts[2]), float(parts[3])))
    return np.array(verts, dtype=np.float32) if verts else None

class ModelNet40Dataset(Dataset):
    """Loads ModelNet40 from PointNet HDF5 files (freely available, no registration)."""
    def __init__(self, data_dir, split='train'):
        files = sorted(glob.glob(f'{data_dir}/ply_data_{split}*.h5'))
        if not files:
            raise FileNotFoundError(f'No HDF5 files at {data_dir}/ply_data_{split}*.h5')
        pcs, labels = [], []
        for f in files:
            with h5py.File(f, 'r') as h:
                pcs.append(h['data'][:, :NUM_POINTS, :3].astype(np.float32))
                labels.append(h['label'][:].flatten().astype(np.int64))
        self.pcs    = np.concatenate(pcs)
        self.labels = np.concatenate(labels)

    def __len__(self): return len(self.pcs)
    def __getitem__(self, i):
        return normalize_pc(torch.from_numpy(self.pcs[i])), int(self.labels[i])

class ScanObjectNNDataset(Dataset):
    """Loads ScanObjectNN OBJ_ONLY split."""
    def __init__(self, split='test'):
        path = f'{SCANOBJECTNN_DIR}/{split}_objectdataset.h5'
        if not os.path.exists(path):
            raise FileNotFoundError(
                f'ScanObjectNN not found at {path}\n'
                'Download from https://hkust-vgd.github.io/scanobjectnn/ '
                'and place at Drive/SACB/data/scanobjectnn/')
        with h5py.File(path, 'r') as h:
            self.pcs    = h['data'][:, :NUM_POINTS, :3].astype(np.float32)
            self.labels = h['label'][:].flatten().astype(np.int64)

    def __len__(self): return len(self.pcs)
    def __getitem__(self, i):
        return normalize_pc(torch.from_numpy(self.pcs[i])), int(self.labels[i])

class ShapeNetCoreDataset(Dataset):
    """Loads ShapeNetCore point clouds from the pre-processed HDF5.
    Build the HDF5 once with the pre-processing cell; loading is then fast."""
    def __init__(self, shapenet_dir):
        h5_path = f'{shapenet_dir}/shapenetcore_pcs.h5'
        if not os.path.exists(h5_path):
            raise FileNotFoundError(
                f'ShapeNetCore HDF5 not found at {h5_path}\n'
                'Run the Downloads cell and then the pre-processing cell first.')
        with h5py.File(h5_path, 'r') as f:
            self.pcs = f['data'][:]          # load fully into RAM: (N, NUM_POINTS, 3)
        print(f'ShapeNetCore: {len(self.pcs):,} point clouds loaded from HDF5')

    def __len__(self): return len(self.pcs)
    def __getitem__(self, i):
        return normalize_pc(torch.from_numpy(self.pcs[i])), 0  # label unused in bridge training

class ContrastivePairDataset(Dataset):
    """Wraps a dataset to return (clean_pc, noisy_pc) pairs."""
    def __init__(self, base, variant):
        self.base = base; self.variant = variant

    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        pc, _ = self.base[i]
        if   self.variant == 'A': noisy = apply_gaussian_noise(pc.unsqueeze(0)).squeeze(0)
        elif self.variant == 'C': noisy = apply_kinect_noise(pc.unsqueeze(0)).squeeze(0)
        else:                     noisy = pc.clone()   # variant B: no noise
        return pc, noisy

def get_dataloaders(variant, num_workers=2):
    # Training source: ShapeNetCore if HDF5 exists, fall back to ModelNet40
    try:
        train_base = ShapeNetCoreDataset(SHAPENET_DIR)
        print('Training source: ShapeNetCore')
    except FileNotFoundError:
        train_base = ModelNet40Dataset(MODELNET40_DIR, 'train')
        print('Training source: ModelNet40 (ShapeNetCore HDF5 not found)')
    train_loader = DataLoader(ContrastivePairDataset(train_base, variant),
                              BATCH_SIZE, shuffle=True, num_workers=num_workers,
                              pin_memory=True, drop_last=True)
    try:
        mn40_test   = ModelNet40Dataset(MODELNET40_DIR, 'test')
        mn40_loader = DataLoader(mn40_test, BATCH_SIZE, shuffle=False, num_workers=num_workers)
    except FileNotFoundError as e:
        print(f'WARNING: ModelNet40 not found — run the Downloads cell. ({e})')
        mn40_loader = None
    try:
        scan_ds = ScanObjectNNDataset('test')
        scan_loader = DataLoader(scan_ds, BATCH_SIZE, shuffle=False, num_workers=num_workers)
    except FileNotFoundError as e:
        print(f'WARNING: {e}'); scan_loader = None
    return train_loader, scan_loader, mn40_loader

print('Datasets defined.')

### Downloads
The following cell will try to **download** the aforementioned datasets. Notice that **ScanObjectNN** requires a **registration** to be downloaded, and ShapenetCore requires HuggingFace access approvale from ShapeNet (gated dataset).

In [ ]:
import shutil

# ── ModelNet40 HDF5 ───────────────────────────────────────────────────────────
if not os.path.exists(f'{MODELNET40_DIR}/ply_data_train0.h5'):
    print('Downloading ModelNet40 HDF5 from HuggingFace…')
    from huggingface_hub import snapshot_download
    tmp = '/tmp/mn40_hf'
    snapshot_download(
        repo_id='Msun/modelnet40',
        repo_type='dataset',
        revision='d5dc795541800feeb7a4b3bd3142729a0d2adf7a',
        local_dir=tmp,
    )
    # Repo contains modelnet40_ply_hdf5_2048.zip — extract H5 files from it
    inner_zip = f'{tmp}/modelnet40_ply_hdf5_2048.zip'
    with zipfile.ZipFile(inner_zip) as z:
        for member in z.namelist():
            if member.endswith('.h5'):
                with z.open(member) as src, \
                     open(f'{MODELNET40_DIR}/{os.path.basename(member)}', 'wb') as dst:
                    dst.write(src.read())
    shutil.rmtree(tmp, ignore_errors=True)
    print('ModelNet40 ready.')
else:
    print('ModelNet40 already downloaded.')

# ── ScanObjectNN ──────────────────────────────────────────────────────────────
if os.path.exists(f'{SCANOBJECTNN_DIR}/test_objectdataset.h5'):
    print('ScanObjectNN found.')
else:
    print('ScanObjectNN not found — attempting download...')
    SCANOBJECTNN_URL = 'https://hkust-vgd.ust.hk/scanobjectnn/h5_files.zip'
    zip_path = os.path.join(SCANOBJECTNN_DIR, 'h5_files.zip')
    try:
        os.makedirs(SCANOBJECTNN_DIR, exist_ok=True)
        print(f'Downloading {SCANOBJECTNN_URL} ...')
        urllib.request.urlretrieve(SCANOBJECTNN_URL, zip_path)
        print('Download complete. Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            targets = {'test_objectdataset.h5', 'training_objectdataset.h5'}
            for member in z.namelist():
                if member.startswith('h5_files/main_split/') and os.path.basename(member) in targets:
                    with z.open(member) as src, \
                         open(os.path.join(SCANOBJECTNN_DIR, os.path.basename(member)), 'wb') as dst:
                        dst.write(src.read())
        os.remove(zip_path)
        print('ScanObjectNN ready.')
    except Exception as e:
        print(f'Auto-download failed: {e}')
        print("""
            To add ScanObjectNN manually:
              1. Download h5_files.zip from https://hkust-vgd.ust.hk/scanobjectnn/h5_files.zip
              2. Extract and upload:
                 MyDrive/SACB/data/scanobjectnn/test_objectdataset.h5
                 MyDrive/SACB/data/scanobjectnn/training_objectdataset.h5
            Then re-run this cell.
            """)

# ── ShapeNetCore ──────────────────────────────────────────────────────────────
# The HuggingFace repo distributes one zip per synset category. We store the
# zips on Drive and defer extraction to the pre-processing cell below, which
# extracts each zip to local /tmp/ and writes a single HDF5 to Drive.
# Writing many small OBJ files directly to Drive would be prohibitively slow.
_shapenet_h5   = f'{SHAPENET_DIR}/shapenetcore_pcs.h5'
_shapenet_zips = sorted(glob.glob(f'{SHAPENET_DIR}/zips/*.zip'))

if os.path.exists(_shapenet_h5):
    print('ShapeNetCore HDF5 found — ready to use.')
elif _shapenet_zips:
    print(f'ShapeNetCore zips found ({len(_shapenet_zips)} synsets). Run the pre-processing cell next.')
else:
    try:
        from google.colab import userdata
        from huggingface_hub import snapshot_download

        HF_TOKEN = userdata.get('HF_TOKEN')
        if not HF_TOKEN:
            raise ValueError('HF_TOKEN not set in Colab Secrets.')

        zip_dir = f'{SHAPENET_DIR}/zips'
        os.makedirs(zip_dir, exist_ok=True)

        print('Downloading ShapeNetCore zip archives from HuggingFace...')
        snapshot_download(
            repo_id='ShapeNet/ShapeNetCore',
            repo_type='dataset',
            local_dir=zip_dir,
            token=HF_TOKEN,
        )
        print('Download complete. Run the pre-processing cell to build the HDF5.')

    except Exception as e:
        print(f'Auto-download failed: {e}')
        print("""
            ShapeNetCore is a gated dataset — access must be requested manually:
              1. Visit https://huggingface.co/datasets/ShapeNet/ShapeNetCore
              2. Submit an access request (name, PI/advisor, affiliation)
              3. Once approved, generate a Read token at https://huggingface.co/settings/tokens
              4. Add it to Colab Secrets (🔑) as HF_TOKEN and re-run this cell.

            Alternatively, place the per-synset zip files at:
              MyDrive/SACB/data/shapenet/zips/{synset_id}.zip
            and run the pre-processing cell directly.
            """)

### ShapeNetCore pre-processing
Extracts each per-synset zip to local Colab disk (`/tmp/`), parses OBJ vertices, subsamples to `NUM_POINTS`, and writes a single compressed HDF5 to Drive. Run once — subsequent sessions skip straight to loading the HDF5.

In [ ]:
_shapenet_h5   = f'{SHAPENET_DIR}/shapenetcore_pcs.h5'
_shapenet_zips = sorted(glob.glob(f'{SHAPENET_DIR}/zips/*.zip'))

if os.path.exists(_shapenet_h5):
    print('ShapeNetCore HDF5 already built — skipping.')
elif not _shapenet_zips:
    print('No ShapeNetCore zips found. Run the Downloads cell first.')
else:
    # Extract each synset zip to local /tmp/ (Colab SSD — fast),
    # parse OBJ vertices, subsample to NUM_POINTS, write one HDF5 to Drive.
    TMP_EXTRACT = '/tmp/shapenet_extract'
    os.makedirs(TMP_EXTRACT, exist_ok=True)

    all_pcs = []
    for zi, zp in enumerate(tqdm(_shapenet_zips, desc='Synsets')):
        synset = os.path.splitext(os.path.basename(zp))[0]
        synset_tmp = f'{TMP_EXTRACT}/{synset}'

        # Extract to local disk — avoids per-file Drive round-trips
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(synset_tmp)

        obj_files = glob.glob(f'{synset_tmp}/**/models/model_normalized.obj', recursive=True)
        for obj in obj_files:
            verts = _load_obj_vertices(obj)
            if verts is None or len(verts) < 3:
                continue
            idx = np.random.choice(len(verts), NUM_POINTS, replace=len(verts) < NUM_POINTS)
            all_pcs.append(verts[idx])

        # Free local disk after each synset
        import shutil
        shutil.rmtree(synset_tmp, ignore_errors=True)

    all_pcs = np.stack(all_pcs).astype(np.float32)   # (N, NUM_POINTS, 3)
    print(f'Writing HDF5: {all_pcs.shape[0]:,} point clouds → {_shapenet_h5}')
    with h5py.File(_shapenet_h5, 'w') as f:
        f.create_dataset('data', data=all_pcs, compression='gzip', compression_opts=4)
    print('ShapeNetCore HDF5 ready.')

## Network

### PointBERT transformer encoder

In [ ]:
class _Mlp(nn.Module):
    def __init__(self, d, hidden=None, drop=0.):
        super().__init__()
        h = hidden or d
        self.net = nn.Sequential(nn.Linear(d,h), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(h,d), nn.Dropout(drop))
    def forward(self, x): return self.net(x)

class _Attention(nn.Module):
    def __init__(self, dim, heads=8, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.heads = heads; self.scale = (dim//heads)**-0.5
        self.qkv   = nn.Linear(dim, dim*3)
        self.drop  = nn.Dropout(attn_drop)
        self.proj  = nn.Linear(dim, dim)
        self.pdrop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        q,k,v = self.qkv(x).reshape(B,N,3,self.heads,C//self.heads).permute(2,0,3,1,4).unbind(0)
        attn  = self.drop((q @ k.transpose(-2,-1)) * self.scale)
        return self.pdrop(self.proj((attn.softmax(-1) @ v).transpose(1,2).reshape(B,N,C)))

class _Block(nn.Module):
    def __init__(self, dim, heads, drop_path=0.):
        super().__init__()
        self.n1  = nn.LayerNorm(dim)
        self.attn = _Attention(dim, heads)
        self.dp  = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        self.n2  = nn.LayerNorm(dim)
        self.mlp = _Mlp(dim, dim*4)

    def forward(self, x):
        x = x + self.dp(self.attn(self.n1(x)))
        return x + self.dp(self.mlp(self.n2(x)))

class PointTransformerEncoder(nn.Module):
    """(B,N,3) -> (B, TRANS_DIM*2) — compatible with ULIP-2 PointBERT weights."""
    def __init__(self):
        super().__init__()
        self.group_divider = Group(NUM_GROUP, GROUP_SIZE)
        self.encoder       = PatchEncoder(ENC_DIMS)
        self.reduce_dim    = nn.Linear(ENC_DIMS, TRANS_DIM)
        self.cls_token     = nn.Parameter(torch.zeros(1, 1, TRANS_DIM))
        self.cls_pos       = nn.Parameter(torch.randn(1, 1, TRANS_DIM))
        self.pos_embed     = nn.Sequential(nn.Linear(3,128), nn.GELU(), nn.Linear(128, TRANS_DIM))
        dpr = torch.linspace(0, DROP_PATH, DEPTH).tolist()
        self.blocks        = nn.ModuleList([_Block(TRANS_DIM, NUM_HEADS, dpr[i]) for i in range(DEPTH)])
        self.norm          = nn.LayerNorm(TRANS_DIM)

    def forward(self, pts):
        nb, center = self.group_divider(pts)
        tok = self.reduce_dim(self.encoder(nb))              # B G D
        B   = pts.shape[0]
        x   = torch.cat([self.cls_token.expand(B,-1,-1), tok], 1)
        pos = torch.cat([self.cls_pos.expand(B,-1,-1), self.pos_embed(center)], 1)
        for blk in self.blocks:
            x = blk(x + pos)
        x = self.norm(x)
        return torch.cat([x[:,0], x[:,1:].max(1)[0]], -1)   # B, D*2

print('PointTransformerEncoder defined.')

### ULIP-2 encoder (3D + text)

In [ ]:
class _QuickGELU(nn.Module):
    """CLIP's original activation — must match pretrained text encoder weights."""
    def forward(self, x): return x * torch.sigmoid(1.702 * x)

class _TextBlock(nn.Module):
    """One CLIP transformer block (batch-first wrapper)."""
    def __init__(self, d, heads, ctx):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, batch_first=False)
        self.ln1  = nn.LayerNorm(d)
        self.mlp  = nn.Sequential(nn.Linear(d, d*4), _QuickGELU(), nn.Linear(d*4, d))
        self.ln2  = nn.LayerNorm(d)
        mask = torch.empty(ctx, ctx).fill_(float('-inf')).triu_(1)
        self.register_buffer('mask', mask)

    def forward(self, x):                                    # x: (B, seq, d)
        a = self.ln1(x).permute(1,0,2)                      # seq-first for MHA
        a = self.attn(a, a, a, attn_mask=self.mask.to(x.dtype))[0].permute(1,0,2)
        x = x + a
        return x + self.mlp(self.ln2(x))

class ULIP2Encoder(nn.Module):
    """Frozen ULIP-2 PointBERT + CLIP text encoder from official checkpoint."""
    def __init__(self, checkpoint_path):
        super().__init__()
        print('Loading ULIP-2 checkpoint…')
        ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        sd   = OrderedDict((k.replace('module.',''), v) for k,v in ckpt['state_dict'].items())

        # 3D point encoder
        self.point_encoder = PointTransformerEncoder()
        pc_sd = {k[len('point_encoder.'):]: v for k,v in sd.items() if k.startswith('point_encoder.')}
        miss, _ = self.point_encoder.load_state_dict(pc_sd, strict=False)
        if miss: print(f'  3D missing keys: {miss[:4]}')
        self.pc_projection = nn.Parameter(sd['pc_projection'].clone())   # (768, 512)

        # Text encoder — infer arch from checkpoint
        tw    = sd['text_projection'].shape[0]        # transformer_width (512 for ViT-B)
        ctx   = sd['positional_embedding'].shape[0]   # context_length (77)
        vs    = sd['token_embedding.weight'].shape[0] # vocab_size (49408)
        depth = sum(1 for k in sd
                    if k.startswith('transformer.resblocks.') and k.endswith('.ln_1.weight'))
        heads = tw // 64
        self.token_embedding      = nn.Embedding(vs, tw)
        self.positional_embedding = nn.Parameter(sd['positional_embedding'].clone())
        self.text_blocks          = nn.Sequential(*[_TextBlock(tw, heads, ctx) for _ in range(depth)])
        self.ln_final             = nn.LayerNorm(tw)
        self.text_projection      = nn.Parameter(sd['text_projection'].clone())
        self.context_length       = ctx

        self.token_embedding.weight.data.copy_(sd['token_embedding.weight'])
        self.ln_final.weight.data.copy_(sd['ln_final.weight'])
        self.ln_final.bias.data.copy_(sd['ln_final.bias'])
        for i, blk in enumerate(self.text_blocks):
            p = f'transformer.resblocks.{i}'
            if f'{p}.ln_1.weight' not in sd: p = f'transformer.{i}'
            blk.ln1.weight.data.copy_(sd[f'{p}.ln_1.weight'])
            blk.ln1.bias.data.copy_(sd[f'{p}.ln_1.bias'])
            blk.ln2.weight.data.copy_(sd[f'{p}.ln_2.weight'])
            blk.ln2.bias.data.copy_(sd[f'{p}.ln_2.bias'])
            blk.attn.in_proj_weight.data.copy_(sd[f'{p}.attn.in_proj_weight'])
            blk.attn.in_proj_bias.data.copy_(sd[f'{p}.attn.in_proj_bias'])
            blk.attn.out_proj.weight.data.copy_(sd[f'{p}.attn.out_proj.weight'])
            blk.attn.out_proj.bias.data.copy_(sd[f'{p}.attn.out_proj.bias'])
            blk.mlp[0].weight.data.copy_(sd[f'{p}.mlp.c_fc.weight'])
            blk.mlp[0].bias.data.copy_(sd[f'{p}.mlp.c_fc.bias'])
            blk.mlp[2].weight.data.copy_(sd[f'{p}.mlp.c_proj.weight'])
            blk.mlp[2].bias.data.copy_(sd[f'{p}.mlp.c_proj.bias'])

        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')
        for p in self.parameters(): p.requires_grad_(False)
        print(f'ULIP-2 ready  |  text depth={depth} heads={heads}  '
              f'3D params={sum(p.numel() for p in self.point_encoder.parameters()):,}')

    @torch.no_grad()
    def encode_pc(self, pc):
        return F.normalize(self.point_encoder(pc) @ self.pc_projection, dim=-1)

    @torch.no_grad()
    def encode_text(self, class_names, descs=None):
        """Ensemble over PROMPT_TEMPLATES. descs overrides CLASS_DESCS lookup."""
        dev = self.pc_projection.device
        d   = descs or [CLASS_DESCS.get(c, c) for c in class_names]
        feats = []
        for tmpl in PROMPT_TEMPLATES:
            toks = self.tokenizer([tmpl.format(x) for x in d]).to(dev)
            x = self.token_embedding(toks) + self.positional_embedding[:toks.shape[1]]
            x = self.text_blocks(x)
            x = self.ln_final(x)
            x = x[torch.arange(len(toks)), toks.argmax(-1)] @ self.text_projection
            feats.append(F.normalize(x, dim=-1))
        return F.normalize(torch.stack(feats).mean(0), dim=-1)

print('ULIP2Encoder defined.')

### ULIP-2 checkpoints download

In [ ]:
# ── ULIP-2 checkpoint ─────────────────────────────────────────────────────────
# Official release: github.com/salesforce/ULIP  (HuggingFace mirror: SFXX/ulip)
# We need the xyz-only (3D) variant — NOT the colored 10k variant.
_ULIP_TARGET = 'ULIP-2-PointBERT-8k-xyz-pc-slip_vit_b-objaverse-pretrained.pt'
_ULIP_HF_REPO = 'SFXX/ulip'

if not os.path.exists(ULIP2_CKPT):
    print('Locating ULIP-2 checkpoint on HuggingFace…')
    # Find the file anywhere in the repo tree
    try:
        all_files = list(list_repo_files(_ULIP_HF_REPO, repo_type='dataset'))
        candidates = [f for f in all_files if _ULIP_TARGET in f]
        if not candidates:
            # Try the simpler name that may appear in some mirrors
            candidates = [f for f in all_files
                          if 'PointBERT' in f and 'ulip' in f.lower() and 'color' not in f.lower()]
        print(f'  Candidates: {candidates}')
    except Exception as e:
        candidates = []
        print(f'  HuggingFace listing failed: {e}')

    if candidates:
        tmp = hf_hub_download(
            repo_id=_ULIP_HF_REPO, repo_type='dataset',
            filename=candidates[0],
            local_dir=f'{BASE}/models/tmp_ulip',
        )
        os.rename(tmp, ULIP2_CKPT)
        print(f'Downloaded → {ULIP2_CKPT}')
    else:
        print("""
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                ULIP-2 checkpoint not found automatically.
                Manual download (one-time, ~400 MB):

                  1. Go to: https://github.com/salesforce/ULIP
                     and follow the download links in the README
                     → look for: ULIP-2-PointBERT-8k-xyz  (xyz-only, NOT colored)

                  2. Upload the .pt file to Google Drive at:
                     MyDrive/SACB/models/ulip2_pointbert.pt

                Then re-run this cell.
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                """)
else:
    print(f'Checkpoint already present.')

### Sensor-Aware Contrastive Bridge

In [ ]:
class SensorAwareContrastiveBridge(nn.Module):
    """3-layer residual MLP: (B,512) noisy features -> (B,512) clean feature space."""
    def __init__(self, d=None):
        super().__init__()
        d = d or FEAT_DIM
        self.net = nn.Sequential(
            nn.Linear(d, d), nn.LayerNorm(d), nn.GELU(),
            nn.Linear(d, d), nn.LayerNorm(d), nn.GELU(),
            nn.Linear(d, d),
        )

    def forward(self, x): return x + self.net(x)
    def count_parameters(self): return sum(p.numel() for p in self.parameters())

print(f'Bridge defined. Params: {SensorAwareContrastiveBridge().count_parameters():,}')

## Training

### Choose variant & load

In [ ]:
VARIANT = 'C'  # ← change to 'A', 'B', or 'C'

encoder = ULIP2Encoder(ULIP2_CKPT).to(device).eval()
train_loader, scan_loader, mn40_loader = get_dataloaders(VARIANT)
print(f'Variant {VARIANT} | train={len(train_loader.dataset)} '
      f'mn40-test={len(mn40_loader.dataset)}'
      + (f' scan-test={len(scan_loader.dataset)}' if scan_loader else ' (no ScanObjectNN)'))

### Train

In [ ]:
trainer = Trainer(VARIANT, encoder)

for epoch in range(1, NUM_EPOCHS + 1):
    loss = trainer.train_epoch(train_loader)
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  loss={loss:.4f}')
    if epoch % 5 == 0 or epoch == NUM_EPOCHS:
        ckpt = trainer.save(epoch)
        print(f'  → {ckpt}')

print('Training complete.')

## Evaluation

### Evaluate


In [ ]:
bridge = trainer.bridge.eval()

scan_base, scan_base_pc, _        = evaluate(None,   encoder, scan_loader,  SCANOBJECTNN_CLASSES)
mn40_base, mn40_base_pc, _        = evaluate(None,   encoder, mn40_loader,  MN40_CLASSES)
scan_br,   scan_br_pc,   scan_cd  = evaluate(bridge, encoder, scan_loader,  SCANOBJECTNN_CLASSES)
mn40_br,   mn40_br_pc,   mn40_cd  = evaluate(bridge, encoder, mn40_loader,  MN40_CLASSES)

print('\n=== Results ===')
if scan_base is not None:
    print(f'ScanObjectNN  baseline: {scan_base:.2f}%   bridge: {scan_br:.2f}%   '
          f'Δ={scan_br-scan_base:+.2f}pp   CD={scan_cd:.4f}')
print(f'ModelNet40    baseline: {mn40_base:.2f}%   bridge: {mn40_br:.2f}%   '
      f'Δ={mn40_br-mn40_base:+.2f}pp   CD={mn40_cd:.4f}')

### Visualizations

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Loss curve
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(range(1, len(trainer.history)+1), trainer.history,
        color='#f58231', lw=2, marker='o', markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('InfoNCE Loss')
ax.set_title(f'Variant {VARIANT} — Training Loss')
ax.yaxis.grid(True, ls='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/loss_variant{VARIANT}.png', dpi=150)
plt.show()

# Per-class accuracy bar chart
if scan_br_pc is not None:
    x = np.arange(len(SCANOBJECTNN_CLASSES)); w = 0.35
    fig, ax = plt.subplots(figsize=(14,5))
    ax.bar(x - w/2, scan_base_pc, w, color='#888',    alpha=0.85, label='Baseline')
    ax.bar(x + w/2, scan_br_pc,   w, color='#f58231', alpha=0.85, label=f'Variant {VARIANT}')
    ax.set_xticks(x); ax.set_xticklabels(SCANOBJECTNN_CLASSES, rotation=35, ha='right')
    ax.set_ylabel('Top-1 Accuracy (%)'); ax.legend()
    ax.set_title('Per-Class Accuracy — ScanObjectNN')
    ax.yaxis.grid(True, ls='--', alpha=0.4); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{RESULTS_DIR}/perclass_variant{VARIANT}.png', dpi=150)
    plt.show()
    print(f'Mean per-class: baseline={scan_base_pc.mean():.1f}%  bridge={scan_br_pc.mean():.1f}%')

### Ablation table
Run after training all three variants (re-run **Training** block and **Evaluate** cell with `VARIANT='A'`, `'B'`, `'C'`).

In [ ]:
results = {}
for var in ['A', 'B', 'C']:
    ckpt_path = f'{CHECKPOINTS_DIR}/sacb_variant{var}_epoch{NUM_EPOCHS:03d}.pt'
    if not os.path.exists(ckpt_path):
        print(f'Variant {var}: checkpoint not found, skipping'); continue
    br = load_bridge(ckpt_path)
    s_acc, _, s_cd = evaluate(br, encoder, scan_loader,  SCANOBJECTNN_CLASSES)
    m_acc, _, m_cd = evaluate(br, encoder, mn40_loader,  MN40_CLASSES)
    results[var] = dict(scan=s_acc, mn40=m_acc, cd=s_cd)

print(f'{"Variant":<12} {"ScanObjNN":>10} {"ModelNet40":>11} {"CD (feat)":>10}')
print('-' * 48)
if scan_base is not None:
    print(f'{"baseline":<12} {scan_base:>9.2f}% {mn40_base:>10.2f}%          —')
for var, r in results.items():
    s = f"{r['scan']:.2f}%" if r['scan'] is not None else '—'
    m = f"{r['mn40']:.2f}%" if r['mn40'] is not None else '—'
    c = f"{r['cd']:.4f}"    if r['cd']   is not None else '—'
    print(f'{var:<12} {s:>10} {m:>11} {c:>10}')